In [1]:
import pandas as pd
import os
import numpy as np
import re

In [2]:
pd.set_option('display.max_columns', None)

## **MO cancer candidate**

In [4]:
file_path = os.path.join("P:\Onedrive\R01-MO-DBT\MO-DBT-data-curation\Data\Cancer\#1", 'mo_cancer_candidate_cohort' + ".xlsx")
mo_cancer_candidate_cohort = pd.read_excel(file_path)

file_path = os.path.join("P:\Onedrive\R01-MO-DBT\MO-DBT-data-curation\Data\Cancer\#1", 'mo_cancer_cohort' + ".xlsx")
mo_cancer_cohort = pd.read_excel(file_path)

In [5]:
mo_cancer_candidate_patients = set(mo_cancer_candidate_cohort['PATIENT_STUDY_ID'].unique())
mo_cancer_patients = set(mo_cancer_cohort['PATIENT_STUDY_ID'].unique())

candidate_patients = mo_cancer_candidate_patients - mo_cancer_patients

In [ ]:
candidate_cohort = mo_cancer_candidate_cohort[mo_cancer_candidate_cohort['PATIENT_STUDY_ID'].isin(candidate_patients)].reset_index(drop=True)

In [7]:
candidate_cohort['PATIENT_STUDY_ID'].nunique()

32

In [50]:
candidate_cohort

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Slab,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index,time_to_INDEX_days,time_to_INDEX_months
0,4333012872,61.0,87158755,2015-10-28,SCREEN,L,NaN,DBT,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,P-Additional projections,2015-10-28,NaN,NaN,NaN,NaN,NaN,NaN
1,4333012872,61.0,87158755,2015-10-28,SCREEN,R,NaN,DBT,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,P-Additional projections,2015-10-28,NaN,NaN,NaN,NaN,NaN,NaN
2,4333012872,62.0,74635440,2016-10-19,DIAG,L,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2016-10-19,NaN,NaN,NaN,NaN,NaN,NaN
3,4333012872,62.0,74635440,2016-10-19,DIAG,R,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2016-10-19,NaN,NaN,NaN,NaN,NaN,NaN
4,4333012872,63.0,79127960,2018-01-30,DIAG,L,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-01-30,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278,4334965484,77.0,88392570,2016-02-24,DIAG,R,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2015-08-26,NaN,NaN,NaN,NaN,NaN,NaN
279,4334965484,78.0,74797146,2017-03-13,DIAG,L,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-03-13,NaN,NaN,NaN,INDEX-1,392.0,12.9
280,4334965484,78.0,74797146,2017-03-13,DIAG,R,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-03-13,NaN,NaN,NaN,INDEX-1,392.0,12.9
281,4334965484,79.0,70929819,2018-04-09,DIAG,L,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-04-09,NaN,NaN,NaN,INDEX,NaN,NaN


## **procedures**

In [11]:
file_path = os.path.join("P:\Onedrive\R01-MO-DBT\MO-DBT-data-curation\Data\Cancer\Cleaned", 'procedures' + ".xlsx")
procedures = pd.read_excel(file_path)

In [9]:
procedure_code = ['3014F', '76090', '77057', '77061', '77062', '77063', '77065', '77065', '77066', '77067', 'G0202']

In [39]:
procedures_subset = procedures[procedures['PROCEDURE_CODE'].isin(procedure_code)]
procedures_subset.reset_index(drop=True, inplace=True)

In [40]:
procedures.shape, procedures_subset.shape

((136810, 7), (73738, 7))

### <span style="color:blue"> **Study**</span> (SCREEN, DIAG)

In [43]:
study_types = {
    "DIAG":   ["DIAG", "DIAGNOSTIC", "DX"],
    "SCREEN": ["SCREENING", "SCREEN"],
}

In [44]:
column_to_check = 'PROCEDURE_NAME'
type_column = 'Study'

procedures_subset[column_to_check] = procedures_subset[column_to_check].astype(str)

for label, terms in study_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = procedures_subset[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    procedures_subset.loc[mask, type_column] = label

C:\Users\tliu\AppData\Local\Temp\ipykernel_15240\517180160.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  procedures_subset[column_to_check] = procedures_subset[column_to_check].astype(str)
C:\Users\tliu\AppData\Local\Temp\ipykernel_15240\517180160.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  procedures_subset.loc[mask, type_column] = label


### <span style="color:blue">**Series**</span> (DBT, IN2D, C VIEW, SECURE)

In [47]:
series_types = {
    "DBT":    ["Breast Tomosynthesis"],
    "FFDM":   ["mammography"],
}

In [48]:
column_to_check = 'PROCEDURE_NAME'
type_column = 'Series'

procedures_subset[column_to_check] = procedures_subset[column_to_check].astype(str)

for label, terms in series_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = procedures_subset[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    procedures_subset.loc[mask, type_column] = label

C:\Users\tliu\AppData\Local\Temp\ipykernel_15240\782023307.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  procedures_subset[column_to_check] = procedures_subset[column_to_check].astype(str)
C:\Users\tliu\AppData\Local\Temp\ipykernel_15240\782023307.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  procedures_subset.loc[mask, type_column] = label


In [49]:
procedures_subset

,PATIENT_STUDY_ID,PROCEDURE_DATE,PROCEDURE_CODE,PROCEDURE_TYPE,PROCEDURE_NAME,PROCEDURE_LOCATION,ORDER_DATE,Study,Series
0,4330018595,2019-08-19,77065,CPT,"Diagnostic mammography, including computer-aid...",NaN,2019-07-29,DIAG,FFDM
1,4330018595,2020-02-28,77065,CPT,"Diagnostic mammography, including computer-aid...",NaN,2020-02-28,DIAG,FFDM
2,4330018595,2020-06-02,77063,CPT,"Screening digital breast tomosynthesis, bilate...",NaN,2020-06-02,SCREEN,DBT
3,4330018595,2020-06-02,77067,CPT,"Screening mammography, bilateral (2-view study...",NaN,2020-06-02,SCREEN,FFDM
4,4330018595,2020-06-10,77065,CPT,"Diagnostic mammography, including computer-aid...",NaN,2020-06-05,DIAG,FFDM
...,...,...,...,...,...,...,...,...,...
73733,4339945522,2020-04-07,77065,CPT,"Diagnostic mammography, including computer-aid...",NaN,2020-04-07,DIAG,FFDM
73734,4339945522,2020-04-13,77065,CPT,"Diagnostic mammography, including computer-aid...",NaN,2020-04-13,DIAG,FFDM
73735,4339959661,2019-07-03,77067,CPT,"Screening mammography, bilateral (2-view study...",NaN,2019-07-03,SCREEN,FFDM
73736,4339959661,2019-09-20,77067,CPT,"Screening mammography, bilateral (2-view study...",NaN,2019-09-20,SCREEN,FFDM


### **Merge**

In [52]:
def merge_procedures(df1, df2, date_col="StudyDate"):
    """
    Attach PROCEDURE_CODE / PROCEDURE_NAME from df2 onto df1.

    Tier 1: PATIENT_STUDY_ID + date + Study + Series   (tight, disambiguates same-day procedures)
    Tier 2: PATIENT_STUDY_ID + date                    (fallback for rows Tier 1 missed,
                                                        e.g. df1 rows where Series is NaN)
    Returns one row per input row of df1.
    """
    left  = df1.copy()
    right = df2.copy()

    left["_d"]  = pd.to_datetime(left[date_col],          errors="coerce").dt.normalize()
    right["_d"] = pd.to_datetime(right["PROCEDURE_DATE"], errors="coerce").dt.normalize()
    left["PATIENT_STUDY_ID"]  = left["PATIENT_STUDY_ID"].astype("int64")
    right["PATIENT_STUDY_ID"] = right["PATIENT_STUDY_ID"].astype("int64")

    keep = ["PATIENT_STUDY_ID", "_d", "Study", "Series", "PROCEDURE_CODE", "PROCEDURE_NAME"]
    right = right[keep].drop_duplicates()

    def collapse(r, keys):
        return (r.groupby(keys, as_index=False)
                 .agg(PROCEDURE_CODE=("PROCEDURE_CODE", lambda s: sorted(set(s.astype(str)))),
                      PROCEDURE_NAME=("PROCEDURE_NAME", lambda s: sorted(set(s.dropna()))),
                      N_PROC=("PROCEDURE_CODE", "size")))

    # --- Tier 1: full key ---
    k1 = ["PATIENT_STUDY_ID", "_d", "Study", "Series"]
    out = left.merge(collapse(right, k1), on=k1, how="left")

    # --- Tier 2: fill rows Tier 1 could not match ---
    k2 = ["PATIENT_STUDY_ID", "_d"]
    miss = out["PROCEDURE_CODE"].isna()
    if miss.any():
        fb = out.loc[miss, k2].merge(collapse(right.drop(columns=["Study", "Series"]), k2),
                                     on=k2, how="left")
        for c in ["PROCEDURE_CODE", "PROCEDURE_NAME", "N_PROC"]:
            out.loc[miss, c] = fb[c].values
        out["MATCH_TIER"] = "study+series"
        out.loc[miss, "MATCH_TIER"] = "date-only"
        out.loc[out["PROCEDURE_CODE"].isna(), "MATCH_TIER"] = "none"
    else:
        out["MATCH_TIER"] = "study+series"

    return out.drop(columns="_d")

In [53]:
merged = merge_procedures(candidate_cohort, procedures_subset, date_col="StudyDate")

In [54]:
print(len(candidate_cohort), "->", len(merged))
print(merged["MATCH_TIER"].value_counts())
print(merged["N_PROC"].value_counts(dropna=False).head())   # how often >1 code per row
merged.loc[merged.MATCH_TIER == "none", ["PATIENT_STUDY_ID","StudyDate","Study","Series"]]

283 -> 283
MATCH_TIER
none         198
date-only     85
Name: count, dtype: int64
N_PROC
NaN    198
1.0     81
2.0      4
Name: count, dtype: int64


,PATIENT_STUDY_ID,StudyDate,Study,Series
0,4333012872,2015-10-28,SCREEN,DBT
1,4333012872,2015-10-28,SCREEN,DBT
2,4333012872,2016-10-19,DIAG,DBT
3,4333012872,2016-10-19,DIAG,DBT
4,4333012872,2018-01-30,DIAG,DBT
...,...,...,...,...
274,4334510859,2020-03-02,DIAG,NaN
275,4334965484,2016-02-24,SCREEN,DBT
276,4334965484,2016-02-24,SCREEN,DBT
277,4334965484,2016-02-24,DIAG,DBT


In [58]:
merged.to_excel("P:\Onedrive\R01-MO-DBT\MO-DBT-data-curation\Data\Cancer\#1\candidate_w_CPT.xlsx", index=False)